使用 Pipeline 和 ColumnTransformer 来对 WA_Fn-UseC_-Telco-Customer-Churn.csv 数据集实现分类

from sklearn.compose import ColumnTransformer

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [40]:
# 1、读取数据
df = pd.read_csv("../../../data/ml/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [41]:
# 2、数据预处理
# 删除不需要的列 customerID
df = df.drop(columns=["customerID"])
# 将原本的字符串数据转化为数值数据
# 先将空字符串和纯空格转换为 NaN，再转换为数值类型
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True),
    errors="coerce",
)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [42]:
# 3、数据拆分
# train:test = 8:2
X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(5634, 19) (5634,)
(1409, 19) (1409,)


In [43]:
# 4、处理标签数据
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)
y_train = pd.DataFrame(y_train, columns=["Churn"])
y_test = pd.DataFrame(y_test, columns=["Churn"])
print(y_train.shape, y_test.shape)

(5634, 1) (1409, 1)


In [44]:
# 5、分离数值数据和类别数据
num_colunms = X_train.select_dtypes(include=[np.number]).columns.tolist()
print("num_colunms: ", num_colunms)
# cate_colunms = df.select_dtypes(include="object").columns.tolist()
cate_colunms = X_train.select_dtypes(include=["str", "object", "category"]).columns.tolist()
print("cate_colunms: ", cate_colunms)

num_colunms:  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
cate_colunms:  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [ ]:
# 6、定义流水线
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cate_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# 使用 ColumnTransformer 来连接两个流水线
processing = ColumnTransformer([
    ("numeric", numeric_pipeline, num_colunms),
    ("categorical", cate_pipeline, cate_colunms),
])

model = Pipeline([
    ("processing", processing),
    ("model", DecisionTreeClassifier(max_features=10, random_state=23))
])

model.fit(X_train, y_train)

In [46]:
# 7、模型评估
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.7260468417317246
              precision    recall  f1-score   support

           0       0.82      0.81      0.81      1040
           1       0.48      0.49      0.48       369

    accuracy                           0.73      1409
   macro avg       0.65      0.65      0.65      1409
weighted avg       0.73      0.73      0.73      1409

[[842 198]
 [188 181]]


In [47]:
# 8. 预测新客户是否会流失
X_validation = X_test[0:50]
y_validation = y_test[0:50]

y_validation_pred = model.predict(X_validation)
y_validation_pred = y_validation_pred.reshape(-1, 1)
res = np.column_stack((y_validation, y_validation_pred, y_validation == y_validation_pred))
print(res)
print(f"accuracy_score: {accuracy_score(y_validation, y_validation_pred)}")
print(f"accuracy_score2: {res[:, -1].sum() / len(y_validation):.2f}")


[[0 0 1]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [1 0 0]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 1 0]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 0 0]
 [0 0 1]
 [0 0 1]
 [1 0 0]
 [0 0 1]
 [1 1 1]
 [1 1 1]
 [0 1 0]
 [1 0 0]
 [0 0 1]]
accuracy_score: 0.84
accuracy_score2: 0.84
